# リカレントニューラルネットワーク

前のモジュールでは、テキストの豊かな意味表現について説明しました。これまで使用してきたアーキテクチャは、文中の単語の集約された意味を捉えますが、単語の<strong>順序</strong>を考慮していません。なぜなら、埋め込みに続く集約操作によって元のテキストからこの情報が失われるためです。これらのモデルは単語の順序を表現できないため、テキスト生成や質問応答のようなより複雑または曖昧なタスクを解決することができません。

文字列の意味を捉えるために、<strong>リカレントニューラルネットワーク</strong>（RNN）と呼ばれるニューラルネットワークのアーキテクチャを使用します。RNNを使う場合、文を一度に一つのトークンずつネットワークに通し、ネットワークは何らかの<strong>状態</strong>を出力します。この状態を次のトークンとともに再びネットワークに入力します。

![Image showing an example recurrent neural network generation.](../../../../../translated_images/ja/rnn.27f5c29c53d727b5.webp)

入力トークン列 $X_0,\dots,X_n$ が与えられると、RNNはニューラルネットワークブロックの列を作成し、この列をエンドツーエンドでバックプロパゲーションを用いて訓練します。各ネットワークブロックは $(X_i,S_i)$ のペアを入力として受け取り、$S_{i+1}$ を出力します。最終状態 $S_n$ または出力 $Y_n$ は線形分類器に渡されて結果が得られます。全てのネットワークブロックは同じ重みを共有しており、1回のバックプロパゲーションでエンドツーエンドに訓練されます。

> 上の図はリカレントニューラルネットワークの展開形（左）とよりコンパクトなリカレント表現（右）を示しています。全てのRNNセルが同じ<strong>共有可能な重み</strong>を持つことを理解することが重要です。

状態ベクトル $S_0,\dots,S_n$ がネットワークを通じて渡されるため、RNNは単語間の順序依存性を学習できます。例えば、シーケンス中に *not* という単語が現れた場合、状態ベクトル内の特定の要素を否定することを学習できます。

内部的には、各RNNセルは2つの重み行列 $W_H$ と $W_I$ 及びバイアス $b$ を持ちます。各RNNステップで、入力 $X_i$ と入力状態 $S_i$ が与えられると、出力状態は $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ と計算されます。ここで $f$ は活性化関数（多くの場合は $\tanh$）です。

> テキスト生成（次のユニットで扱います）や機械翻訳のような問題では、各RNNステップで何らかの出力値も得たい場合があります。この場合は別の行列 $W_O$ も存在し、出力は $Y_i=f(W_O\times S_i+b_O)$ と計算されます。

リカレントニューラルネットワークがどのようにニュースデータセットの分類に役立つか見てみましょう。

> サンドボックス環境では、次のセルを実行して必要なライブラリがインストールされており、データがプリフェッチされていることを確認する必要があります。ローカルで実行している場合は次のセルはスキップしても構いません。


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

大規模なモデルをトレーニングする際、GPUメモリの割り当てが問題になることがあります。また、データがGPUメモリに収まるようにしつつトレーニングが十分速くなるように、異なるミニバッチサイズで実験する必要があるかもしれません。ご自身のGPUマシンでこのコードを実行している場合は、トレーニングの高速化のためにミニバッチサイズの調整を試みることができます。

> <strong>注意</strong>: 特定のNVidiaドライバのバージョンでは、モデルのトレーニング後にメモリが解放されないことが知られています。このノートブックでいくつかの例を実行しており、特に同じノートブック内で独自の実験を行っている場合などに、メモリが枯渇する可能性があります。モデルのトレーニング開始時に奇妙なエラーが発生した場合は、ノートブックのカーネルを再起動するとよいでしょう。


In [3]:
batch_size = 16
embed_size = 64

## シンプルRNN分類器

シンプルなRNNの場合、各再帰ユニットは入力ベクトルと状態ベクトルを受け取り、新しい状態ベクトルを出力するシンプルな線形ネットワークです。Kerasでは、これは`SimpleRNN`レイヤーで表現できます。

ワンホットエンコードされたトークンを直接RNNレイヤーに渡すこともできますが、その高次元性のためあまり良い方法ではありません。したがって、単語ベクトルの次元を下げるために埋め込みレイヤーを使い、その後にRNNレイヤー、最後に`Dense`分類器を使います。

> **Note**: 次元がそれほど高くない場合、例えば文字レベルのトークン化を使う場合は、ワンホットエンコードされたトークンをRNNセルに直接渡すのが理にかなっていることがあります。


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **注意:** ここでは簡単のために未学習の埋め込み層を使用していますが、より良い結果を得るためには、前のユニットで説明したようにWord2Vecを用いた事前学習済みの埋め込み層を使用することができます。このコードを事前学習済みの埋め込みに対応させるのは良い練習になるでしょう。

それではRNNを訓練しましょう。一般にRNNは訓練が非常に難しいです。なぜなら、シーケンス長に沿ってRNNセルを展開すると、逆伝播に関与する層の数が非常に多くなるためです。そのため、小さな学習率を選択し、より大きなデータセットでネットワークを訓練して良い結果を出す必要があります。これにはかなり時間がかかることがあるので、GPUの使用が推奨されます。

処理を高速化するために、説明文は省略してニュースのタイトルだけでRNNモデルを訓練します。説明文ありで訓練して、モデルが訓練できるか試してみるのも良いでしょう。


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>注意</strong> ここではニュースの見出しのみを訓練しているため、精度は低くなる可能性があります。


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


マスキングを使っているので、タイトルと説明文の全データセットでモデルを訓練できます。

> **Note**: ニュースのタイトルのみで訓練されたベクトライザーを使っていて、記事全体の本文ではないことに気づきましたか？このため、いくつかのトークンが無視される可能性があるので、ベクトライザーを再訓練したほうが良いかもしれません。ただし、影響はごくわずかかもしれないので、簡単さのために以前の事前訓練済みベクトライザーを使い続けます。


## LSTM: 長短期記憶

RNNの主な問題のひとつは<strong>勾配消失</strong>です。RNNはかなり長くなることがあり、逆伝播の際に勾配をネットワークの最初の層まで伝えるのが難しくなることがあります。これが起こると、ネットワークは離れたトークン間の関係を学習できません。この問題を回避する一つの方法は、<strong>ゲート</strong>を使って<strong>明示的な状態管理</strong>を導入することです。ゲートを導入する最も一般的な2つのアーキテクチャは、<strong>長短期記憶</strong>（LSTM）と<strong>ゲート付きリレー単位</strong>（GRU）です。ここではLSTMについて説明します。

![長短期記憶セルの例を示す画像](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTMネットワークはRNNと似た構造ですが、レイヤー間で渡される状態が2つあります：実際の状態$c$と隠れベクトル$h$です。各ユニットでは、隠れベクトル$h_{t-1}$が入力$x_t$と組み合わされ、<strong>ゲート</strong>によって状態$c_t$と出力$h_t$が制御されます。各ゲートはシグモイド活性化（出力は$[0,1]$の範囲）を持ち、これは状態ベクトルに乗算されるとビットマスクのように考えられます。LSTMには以下のゲートがあります（上の図の左から右へ）：
* <strong>忘却ゲート</strong>はベクトル$c_{t-1}$のどの成分を忘れ、どの成分を通すかを決定します。
* <strong>入力ゲート</strong>は入力ベクトルと前の隠れベクトルからどれだけの情報を状態ベクトルに取り込むかを決定します。
* <strong>出力ゲート</strong>は新しい状態ベクトルから新しい隠れベクトル$h_t$を生成する際に使用される成分を決定します。

状態$c$の成分は、オン・オフを切り替えられるフラグのように考えられます。例えば、シーケンス内で名前の<em>Alice</em>に出会うと、それが女性を指すと推測して女性名詞のフラグを状態に立てます。さらに<em>and Tom</em>という単語に出会うと、複数名詞のフラグを立てます。このように状態を操作することで文法的な性質を追跡できます。

> **Note**: LSTMの内部構造を理解するのに役立つ素晴らしいリソースはこちらです：[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)（Christopher Olah著）。

LSTMセルの内部構造は複雑に見えますが、Kerasでは`LSTM`レイヤーの中にこの実装が隠されているので、上の例では再帰レイヤーを置き換えるだけで済みます。


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>注意</strong> LSTMのトレーニングもかなり遅くなることがあり、トレーニングの初期段階ではあまり精度の向上が見られないかもしれません。良い精度を達成するには、しばらくトレーニングを続ける必要がある場合があります。


## 双方向および多層RNN

これまでの例では、リカレントネットワークはシーケンスの開始から終了まで一方向に動作していました。これは私たちが読む方向や音声を聞く方向と同じため自然に感じられます。しかし、入力シーケンスをランダムにアクセスする必要がある場合は、リカレント計算を両方向で行う方が理にかなっています。両方向で計算を行えるRNNは<strong>双方向</strong>RNNと呼ばれ、特殊な`Bidirectonal`レイヤーでリカレントレイヤーをラップすることで作成できます。

> <strong>注</strong>: `Bidirectional`レイヤーは内包するレイヤーのコピーを2つ作成し、そのうちの1つの`go_backwards`プロパティを`True`に設定して、シーケンスに対して逆方向に動作させます。

リカレントネットワークは一方向でも双方向でも、シーケンス内のパターンを捉えて状態ベクトルに格納するか、出力として返します。畳み込みネットワークと同様に、最初のレイヤーで抽出された低レベルのパターンから上位レベルのパターンを捉えるために、最初のレイヤーに続けて別のリカレントレイヤーを構築できます。これが、前のレイヤーの出力を次のレイヤーの入力として渡す、2層以上のリカレントネットワークからなる<strong>多層RNN</strong>という概念につながります。

![Multilayer long-short-term-memory- RNNの画像](../../../../../translated_images/ja/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Fernando Lópezによる[この素晴らしい投稿](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)からの図です。*

Kerasでは、単にモデルにリカレントレイヤーを増やすだけでこれらのネットワークを簡単に構築できます。最後のレイヤーを除いて、すべてのレイヤーに`return_sequences=True`パラメータを指定する必要があります。これは、リカレント計算の最終状態だけでなく、中間のすべての状態を返すためです。

分類問題のために、双方向の2層LSTMを構築してみましょう。

> <strong>注</strong> このコードも実行にかなり時間がかかりますが、これまでで最も高い精度を得られます。結果を見るために待つ価値があるかもしれません。


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## 他のタスクのためのRNN

これまでは、RNNを使ってテキストのシーケンスを分類することに焦点を当ててきました。しかし、RNNはテキスト生成や機械翻訳など、他にも多くのタスクを処理できます。これらのタスクについては次のユニットで考えます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
